# Whisper LoRA 파인튜닝 — 금융 상담 음성 (H200)

**목적**: KtelSpeech 금융 상담 음성으로 Whisper를 도메인 적응

**핵심**: 그냥 학습만 하지 않고 **파인튜닝 전/후 CER을 측정**해 개선 수치를 남긴다.
→ "돌려봤다"가 아니라 "CER X% → Y%, N% 개선"으로 말하기 위함

**실행 순서**

1. 셀 1 실행 → **Kernel Restart**
2. 셀 2~4 (baseline CER — 반드시 학습 전에)
3. 셀 5~7 (LoRA 학습)
4. 셀 8 (개선폭 확인)

## 1. 추가 패키지 설치

In [3]:
# =====================================================================
# [추가 패키지] 오디오 로드 + 평가 지표
# ---------------------------------------------------------------------
# librosa, soundfile : wav 로드 및 16kHz 리샘플링 (Whisper 입력 요구사항)
# jiwer              : CER(문자오류율) 계산 - 한국어는 WER보다 CER이 적합
# ※ 공용 서버라 --user 필수 (공용 폴더 쓰기 권한 없음)
# =====================================================================
%pip install --user librosa soundfile jiwer --quiet --no-warn-script-location
print("완료 - Kernel > Restart Kernel 후 다음 셀 진행")

Note: you may need to restart the kernel to use updated packages.
완료 - Kernel > Restart Kernel 후 다음 셀 진행


## 2. 설정 및 데이터 확인

팀원은 이 셀의 상수만 수정하면 됩니다.

In [1]:
# =====================================================================
# [설정] 경로·모델·하이퍼파라미터를 한 곳에 모아둠
# =====================================================================
import os, json, torch

# --- GPU 지정 (공용 서버: 배정받은 device 번호로 변경) ---
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

DATA_ROOT  = "./ktel_data"                          # 압축 푼 데이터 폴더
BASE_MODEL = "seastar105/whisper-medium-komixv2"    # 한국어+전화망 사전학습 Whisper
OUT_DIR    = "./whisper-lora-finance"               # 학습 결과 저장

# --- 하이퍼파라미터 ---
BATCH_SIZE    = 16      # H200 140GB라 여유. OOM 나면 8로
LEARNING_RATE = 1e-3    # LoRA는 풀파인튜닝보다 큰 LR 사용
NUM_EPOCHS    = 3
EVAL_SAMPLES  = 200     # CER 측정용 valid 샘플 수 (전체 500 중 일부, 시간 절약)

# --- 데이터 존재 확인 ---
for split in ["train", "valid"]:
    with open(f"{DATA_ROOT}/{split}.jsonl", encoding="utf-8") as f:
        lines = f.readlines()
    first = json.loads(lines[0])
    wav_ok = os.path.exists(os.path.join(DATA_ROOT, first["audio"]))
    print(f"[{split}] {len(lines)}건 | wav 연결: {wav_ok} | 예시: {first['text'][:30]}...")

print("GPU:", torch.cuda.get_device_name(0))

[train] 8000건 | wav 연결: True | 예시: 그럼 아이들은 어떻게 되는 거 어떻게 되지요? 아이들은...
[valid] 500건 | wav 연결: True | 예시: 신랑은 젊게 살아서 어플로 모든 걸 다하더라고요. 그럼...
GPU: NVIDIA H200 NVL


## 3.soundfile 

In [3]:
# torchcodec 설치 실패 시 대안: 이전 방식(soundfile) 쓰는 버전으로
%pip install --user "datasets<4.0" --quiet --no-warn-script-location

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
trl 1.9.0 requires datasets>=4.7.0, but you have datasets 3.6.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


## 3. 데이터셋 로드

In [2]:
# =====================================================================
# [데이터셋] jsonl -> HuggingFace Dataset
# ---------------------------------------------------------------------
# Whisper는 16kHz 모노 오디오를 입력으로 받으므로 Audio(16000)로 캐스팅
# =====================================================================
from datasets import load_dataset, Audio

data = load_dataset("json", data_files={
    "train": f"{DATA_ROOT}/train.jsonl",
    "valid": f"{DATA_ROOT}/valid.jsonl",
})

# jsonl의 audio는 DATA_ROOT 기준 상대경로 -> 절대경로로 변환
def to_abs(batch):
    batch["audio"] = os.path.join(DATA_ROOT, batch["audio"])
    return batch

data = data.map(to_abs)
data = data.cast_column("audio", Audio(sampling_rate=16000))

print(data)
print("샘플:", data["train"][0]["text"][:50])

Generating train split: 0 examples [00:00, ? examples/s]

Generating valid split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['audio', 'text'],
        num_rows: 8000
    })
    valid: Dataset({
        features: ['audio', 'text'],
        num_rows: 500
    })
})
샘플: 그럼 아이들은 어떻게 되는 거 어떻게 되지요? 아이들은 따로 항공권을 끊어야 하나요? 제가


## 4. 모델 로드 + baseline CER 측정 (중요)

**반드시 학습 전에 실행.** 학습 후에 재면 baseline이 오염되어 개선폭을 말할 수 없다.

In [3]:
# =====================================================================
# [BASELINE] 파인튜닝 '전' 성능 측정 - 이게 있어야 개선폭을 말할 수 있다
# ---------------------------------------------------------------------
# CER(Character Error Rate): 낮을수록 좋음.
# 한국어는 어절 단위 WER보다 문자 단위 CER이 인식 품질을 더 잘 반영
# =====================================================================
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import jiwer

processor = WhisperProcessor.from_pretrained(BASE_MODEL, language="korean", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16
).to("cuda")
model.generation_config.language = "korean"
model.generation_config.task = "transcribe"

def measure_cer(model, n=EVAL_SAMPLES, tag=""):
    """valid 셋 앞 n개로 CER 측정"""
    model.eval()
    preds, refs = [], []
    for i in range(n):
        ex = data["valid"][i]
        inputs = processor(
            ex["audio"]["array"], sampling_rate=16000, return_tensors="pt"
        ).input_features.to("cuda", torch.float16)
        with torch.no_grad():
            ids = model.generate(inputs, max_new_tokens=200)
        preds.append(processor.batch_decode(ids, skip_special_tokens=True)[0].strip())
        refs.append(ex["text"].strip())
        if (i + 1) % 50 == 0:
            print(f"  {tag} {i+1}/{n} 진행...")
    return jiwer.cer(refs, preds), preds, refs

print("파인튜닝 전(baseline) CER 측정 중... (수 분 소요)")
cer_before, preds_b, refs_b = measure_cer(model, tag="[before]")
print("")
print(f"BASELINE CER = {cer_before:.4f} ({cer_before*100:.2f}%)")

# 어떤 식으로 틀리는지 눈으로 확인
for i in range(3):
    print("")
    print("정답:", refs_b[i])
    print("예측:", preds_b[i])

preprocessor_config.json:   0%|          | 0.00/339 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.29k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

...

  [before] 100/200 진행...


[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  [before] 150/200 진행...


[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  [before] 200/200 진행...

BASELINE CER = 0.1170 (11.70%)

정답: 신랑은 젊게 살아서 어플로 모든 걸 다하더라고요. 그럼 어플로 하라고 해야겠어요. 제 아파트 관리비 자동이체는 그냥 해지해 주시겠어요?
예측: 신랑은 젊게 살아도 어플로 모든 걸 다 하더라고요 그럼 어플로 하라고 해야겠어요 제가 다트 관리비 저도 지금 그냥 해지해 주시겠어요?

정답: 저는 김창출이고요.주민번호는 칠 오 공 이 이 삼 일 이 삼 사 오 육 칠입니다.
예측: 저는 김창추리구요 주민번호는 7 5 0 2 2 3 1 2 3 4 5 6 7 입니다.

정답: 그럼 티 피 엑스 주식은 다시 살아날 가망이 있을까요? 있든 없든 그 전산 오류 전 금액으로 오 성 증권사에서 저희 거 매수 해주시던 해야 하는 거 아니에요? 다른 증권사는 그렇게도 하던데요. 한 번 방법 알아보세요.
예측: 그럼 TTS 주식은 다시 살아날 가능성은 있을까요 있든 없든 그 전산오류 전 금액으로 우선 증권사에서 저희 거 매수해 주시던 해야 하는 거 아니에요 다른 증권사는 그렇기도 하던데요 저 한번 방법 알아보세요.


## 5. LoRA 설정

In [4]:
# =====================================================================
# [LoRA] 왜 풀파인튜닝이 아닌가
# ---------------------------------------------------------------------
# - 8천건 규모에 풀파인튜닝은 과적합·원본 성능 손상(catastrophic forgetting) 위험
# - LoRA는 어텐션 일부만 학습해 가볍고, 원본 가중치를 보존하며,
#   학습이 빨라 하이퍼파라미터 실험을 여러 번 돌릴 수 있다
# =====================================================================
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=32,                                  # rank: 클수록 표현력 증가, 파라미터 증가
    lora_alpha=64,
    target_modules=["q_proj", "v_proj"],   # 어텐션 Q,V에만 적용 (Whisper 표준)
    lora_dropout=0.05,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()   # 전체 대비 학습 파라미터 비율

trainable params: 9,437,184 || all params: 773,295,104 || trainable%: 1.2204


## 6. 전처리 + Collator

In [8]:
# =====================================================================
# [전처리] 오디오 -> log-Mel 특징, 텍스트 -> 토큰 ID
# ---------------------------------------------------------------------
# data는 Audio(16000)로 캐스팅된 상태이므로 batch["audio"]는
# {'path','array','sampling_rate'} dict. array를 그대로 사용한다.
# =====================================================================
def prepare(batch):
    audio = batch["audio"]
    batch["input_features"] = processor.feature_extractor(
        audio["array"], sampling_rate=16000
    ).input_features[0]
    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    return batch

data_proc = data.map(prepare, remove_columns=data["train"].column_names)
# ---------------------------------------------------------------------
# Collator: 길이가 다른 샘플을 배치로 묶을 때 패딩 처리
# 패딩 토큰은 loss 계산에서 제외(-100)해야 학습이 왜곡되지 않는다
# ---------------------------------------------------------------------
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2Seq:
    processor: Any
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]):
        input_feats = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_feats, return_tensors="pt")
        label_feats = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_feats, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch["labels"] = labels
        return batch

collator = DataCollatorSpeechSeq2Seq(processor=processor)
print("전처리 완료")

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

전처리 완료


## 7. 학습

주의: 서버 세션이 24시간이므로 학습이 그 안에 끝나야 함.
8천건 x 3epoch은 H200에서 약 1~2시간 예상.

In [9]:
# =====================================================================
# [학습] LoRA 파인튜닝 실행
# =====================================================================
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

args = Seq2SeqTrainingArguments(
    output_dir=OUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    fp16=True,
    logging_steps=25,
    save_strategy="epoch",
    report_to=[],              # wandb 미사용
    remove_unused_columns=False,
    label_names=["labels"],
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=data_proc["train"],
    data_collator=collator,
)

trainer.train()
model.save_pretrained(f"{OUT_DIR}/final")
print("학습 완료 ->", f"{OUT_DIR}/final")

Step,Training Loss
25,0.992880
50,0.292090
75,0.280525
100,0.290795
125,0.262756
150,0.236912
175,0.270741
200,0.228218
225,0.269022
250,0.242045


학습 완료 -> ./whisper-lora-finance/final


## 8. 학습 후 CER 측정 + 개선폭

이 수치가 이번 파인튜닝의 성과 지표.

In [10]:
# =====================================================================
# [결과] 파인튜닝 후 CER 측정 -> 개선폭 산출
# =====================================================================
print("파인튜닝 후 CER 측정 중...")
cer_after, preds_a, refs_a = measure_cer(model, tag="[after]")

improve = (cer_before - cer_after) / cer_before * 100

print("")
print("=" * 55)
print(f"  파인튜닝 전 CER : {cer_before*100:.2f}%")
print(f"  파인튜닝 후 CER : {cer_after*100:.2f}%")
print(f"  상대 개선폭      : {improve:.1f}%")
print("=" * 55)

# 발표 자료용 전/후 비교
print("")
print("[전/후 비교 샘플]")
for i in range(5):
    print("")
    print("정답 :", refs_a[i])
    print("전   :", preds_b[i])
    print("후   :", preds_a[i])

# 결과 파일 저장 (문서화용)
with open(f"{OUT_DIR}/cer_result.json", "w", encoding="utf-8") as f:
    json.dump({
        "base_model": BASE_MODEL,
        "train_samples": len(data["train"]),
        "eval_samples": EVAL_SAMPLES,
        "cer_before": cer_before,
        "cer_after": cer_after,
        "relative_improvement_pct": improve,
    }, f, ensure_ascii=False, indent=2)
print("")
print("결과 저장:", f"{OUT_DIR}/cer_result.json")

[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


파인튜닝 후 CER 측정 중...


[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  [after] 50/200 진행...


[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  [after] 100/200 진행...


[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  [after] 150/200 진행...


[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  [after] 200/200 진행...

  파인튜닝 전 CER : 11.70%
  파인튜닝 후 CER : 6.92%
  상대 개선폭      : 40.8%

[전/후 비교 샘플]

정답 : 신랑은 젊게 살아서 어플로 모든 걸 다하더라고요. 그럼 어플로 하라고 해야겠어요. 제 아파트 관리비 자동이체는 그냥 해지해 주시겠어요?
전   : 신랑은 젊게 살아도 어플로 모든 걸 다 하더라고요 그럼 어플로 하라고 해야겠어요 제가 다트 관리비 저도 지금 그냥 해지해 주시겠어요?
후   : 신랑은 전계사라도 어플로 모든 걸 다 하더라고요. 그럼 어플로 하라고 해야겠어요. 제한 박스 관리 계좌로 이쪽 그냥 해지해주시겠어요?

정답 : 저는 김창출이고요.주민번호는 칠 오 공 이 이 삼 일 이 삼 사 오 육 칠입니다.
전   : 저는 김창추리구요 주민번호는 7 5 0 2 2 3 1 2 3 4 5 6 7 입니다.
후   : 저는 김 김창출이구요. 주민번호는 칠 오 공 이 이 삼 일 이 삼 사 오 육 칠입니다.

정답 : 그럼 티 피 엑스 주식은 다시 살아날 가망이 있을까요? 있든 없든 그 전산 오류 전 금액으로 오 성 증권사에서 저희 거 매수 해주시던 해야 하는 거 아니에요? 다른 증권사는 그렇게도 하던데요. 한 번 방법 알아보세요.
전   : 그럼 TTS 주식은 다시 살아날 가능성은 있을까요 있든 없든 그 전산오류 전 금액으로 우선 증권사에서 저희 거 매수해 주시던 해야 하는 거 아니에요 다른 증권사는 그렇기도 하던데요 저 한번 방법 알아보세요.
후   : 그럼 티 피 에스 주식은 다시 살아날 가망은 있을까요? 있든 없든 전산 오류 전 금액으로 우성 증권사에서 저희거 매수해주시던 해야 하는 거 아니에요? 다른 증권사는 그렇기도 하던데요. 한 번 방법 알아보세요.

정답 : 네 다름이 아니고 제가 주식에 대해서 좀 공부를 해볼까 하는데 인터넷에 보니까 먼저 증권계좌를 개설해야 한다고 하더라구요.
전   : 네 다름이 아니고 제가 주식에 대해서 좀 공부를 해볼까 하는데 인터넷에

## 트러블슈팅

| 증상 | 대응 |
|---|---|
| CUDA OOM | `BATCH_SIZE` 16 → 8 → 4 |
| 학습이 24h 넘을 듯 | `NUM_EPOCHS` 축소 또는 train 샘플 축소 |
| CER이 오히려 나빠짐 | `LEARNING_RATE` 1e-3 → 5e-4, epoch 축소 |
| 다른 사람 GPU 점유 | `CUDA_VISIBLE_DEVICES` 배정 번호 확인 |
| torch CUDA 인식 실패 | torch를 최신으로 재설치하지 말 것 (cu126 빌드 사용) |

In [12]:
model.save_pretrained("./whisper-lora-finance/final")
print("저장 완료")
import os
print(os.listdir("./whisper-lora-finance/final"))

현재 작업 디렉토리: /home/j-i15a708

[현재 폴더 내용]
     .bash_history
     .bash_logout
     .bashrc
  📁 .cache
  📁 .conda
  📁 .ipynb_checkpoints
  📁 .ipython
  📁 .jupyter
  📁 .local
  📁 .nv
     .profile
     Pinetuning_setting.ipynb
     ktel_prepared.zip
  📁 stt-finetune

[홈에서 adapter 파일 검색]
/home/j-i15a708/.cache/huggingface/hub/models--seastar105--whisper-medium-komixv2/.no_exist/752b1b5bfa0c8219bca737ea180a4ad7eed1e621/adapter_config.json
/home/j-i15a708/stt-finetune/whisper-lora-finance/checkpoint-500/adapter_config.json
/home/j-i15a708/stt-finetune/whisper-lora-finance/checkpoint-500/adapter_model.safetensors
/home/j-i15a708/stt-finetune/whisper-lora-finance/final/adapter_config.json
/home/j-i15a708/stt-finetune/whisper-lora-finance/final/adapter_model.safetensors
/home/j-i15a708/stt-finetune/whisper-lora-finance/checkpoint-1000/adapter_config.json
/home/j-i15a708/stt-finetune/whisper-lora-finance/checkpoint-1000/adapter_model.safetensors
/home/j-i15a708/stt-finetune/whisper-lora-finance/

In [16]:
# =====================================================================
# [경로] 노트북이 폴더 이동돼 커널의 작업 디렉토리가 옛 위치를 가리킴.
#        모듈을 찾도록 현재 노트북 폴더를 sys.path에 추가한다.
# =====================================================================
import sys, os
from pathlib import Path

PROJ = str(Path.home() / "stt-finetune")
os.chdir(PROJ)              # 작업 디렉토리도 프로젝트로 이동
if PROJ not in sys.path:
    sys.path.insert(0, PROJ)

print("작업 디렉토리:", os.getcwd())
print("exp_tracker 존재:", os.path.exists("exp_tracker.py"))

작업 디렉토리: /home/j-i15a708/stt-finetune
exp_tracker 존재: True


In [17]:
# =====================================================================
# [실험 기록] exp1 결과를 로그에 남기고 리포트 생성
# ---------------------------------------------------------------------
# exp_tracker.py가 이 노트북과 같은 폴더에 있어야 import 된다.
# =====================================================================
from exp_tracker import ExpTracker

tracker = ExpTracker()

# 이번 실험의 설정을 명시적으로 기록 (무엇을 바꿨는지가 핵심)
tracker.start(
    "exp1_lora_r32_ep3",
    config={
        "base": BASE_MODEL,
        "lora_r": 32,
        "target": "q_proj,v_proj",
        "lr": LEARNING_RATE,
        "epochs": NUM_EPOCHS,
        "train_n": 8000,
        "eval": "없음",
    },
    note="첫 LoRA 실험. 숫자표기·구두점 크게 개선. 일부 샘플 퇴화로 과적합 의심"
)

# 결과 기록 (오류 분해 + 개선/퇴화 샘플 자동 분석)
tracker.finish(cer_before, cer_after, refs_a, preds_b, preds_a)

# 포폴용 마크다운 리포트 생성
tracker.report()

[exp] exp1_lora_r32_ep3 시작
      config: {"base": "seastar105/whisper-medium-komixv2", "lora_r": 32, "target": "q_proj,v_proj", "lr": 0.001, "epochs": 3, "train_n": 8000, "eval": "없음"}

  실험      : exp1_lora_r32_ep3
  소요      : 0.0분
  CER       : 11.7%  ->  6.92%  (40.8% 개선)
  오류 분해 : 구두점 0.51%p / 숫자표기 0.05%p / 띄어쓰기 0.4%p
              순수 인식 오류 5.97%
  샘플      : 개선 112건 / 퇴화 35건
  ⚠ 퇴화 비중 높음 - 과적합 의심 (epoch 축소/eval 도입 검토)
리포트 생성 완료: ./experiments/REPORT.md


'./experiments/REPORT.md'

In [18]:
# =====================================================================
# [exp2] 검증 도입 + best checkpoint 선택
# ---------------------------------------------------------------------
# exp1 문제: 3 epoch 끝까지 학습 후 마지막 상태 사용 -> 퇴화 35건(과적합)
# 가설: epoch마다 valid loss를 재고 가장 좋은 시점을 채택하면 퇴화 감소
# 원칙: 한 번에 변수 하나만 바꾼다 (eval 도입 외 나머지는 exp1과 동일)
# =====================================================================
from transformers import WhisperForConditionalGeneration, Seq2SeqTrainer, Seq2SeqTrainingArguments
from peft import LoraConfig, get_peft_model
import torch

# 원본 모델 새로 로드 (exp1 학습 결과가 섞이면 비교가 무의미해짐)
model2 = WhisperForConditionalGeneration.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16).to("cuda")
model2.generation_config.language = "korean"
model2.generation_config.task = "transcribe"

model2 = get_peft_model(model2, LoraConfig(
    r=32, lora_alpha=64,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05, bias="none",
))
model2.print_trainable_parameters()

args2 = Seq2SeqTrainingArguments(
    output_dir="./whisper-lora-exp2",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    fp16=True,
    logging_steps=25,
    # --- exp1과의 유일한 차이 ---
    eval_strategy="epoch",           # epoch마다 검증
    save_strategy="epoch",
    load_best_model_at_end=True,     # 가장 좋은 시점 자동 채택
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,              # 디스크 절약
    # ---------------------------
    report_to=[],
    remove_unused_columns=False,
    label_names=["labels"],
)

trainer2 = Seq2SeqTrainer(
    model=model2,
    args=args2,
    train_dataset=data_proc["train"],
    eval_dataset=data_proc["valid"],   # exp1엔 없던 부분
    data_collator=collator,
)

trainer2.train()
model2.save_pretrained("./whisper-lora-exp2/final")
print("exp2 학습 완료")

Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

trainable params: 9,437,184 || all params: 773,295,104 || trainable%: 1.2204


Epoch,Training Loss,Validation Loss
1,0.237197,0.250992
2,0.147438,0.267537
3,0.049337,0.321121


exp2 학습 완료


In [19]:
# =====================================================================
# [exp2 평가] 동일한 valid 샘플로 CER 측정 (exp1과 조건 동일해야 비교 가능)
# =====================================================================
cer_after2, preds_a2, refs_a2 = measure_cer(model2, tag="[exp2]")
print(f"exp2 CER = {cer_after2*100:.2f}%")

[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  [exp2] 50/200 진행...


[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  [exp2] 100/200 진행...


[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  [exp2] 150/200 진행...


[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  [exp2] 200/200 진행...
exp2 CER = 6.72%


In [20]:
# =====================================================================
# [exp2 기록] 로그에 누적 -> REPORT.md에 2행째가 쌓인다
# =====================================================================
tracker.start("exp2_eval_bestckpt",
    config={"base": BASE_MODEL, "lora_r": 32, "target": "q_proj,v_proj",
            "lr": LEARNING_RATE, "epochs": NUM_EPOCHS, "train_n": 8000,
            "eval": "epoch+best_ckpt"},
    note="exp1 대비 eval/best checkpoint만 추가. 퇴화 샘플 감소 여부 확인")
tracker.finish(cer_before, cer_after2, refs_a2, preds_b, preds_a2)
tracker.report()

[exp] exp2_eval_bestckpt 시작
      config: {"base": "seastar105/whisper-medium-komixv2", "lora_r": 32, "target": "q_proj,v_proj", "lr": 0.001, "epochs": 3, "train_n": 8000, "eval": "epoch+best_ckpt"}

  실험      : exp2_eval_bestckpt
  소요      : 0.0분
  CER       : 11.7%  ->  6.72%  (42.6% 개선)
  오류 분해 : 구두점 0.46%p / 숫자표기 -0.01%p / 띄어쓰기 0.33%p
              순수 인식 오류 5.94%
  샘플      : 개선 125건 / 퇴화 29건
리포트 생성 완료: ./experiments/REPORT.md


'./experiments/REPORT.md'

In [21]:
tracker.start("exp2_eval_bestckpt",
    config={"base": BASE_MODEL, "lora_r": 32, "target": "q_proj,v_proj",
            "lr": LEARNING_RATE, "epochs": NUM_EPOCHS, "train_n": 8000,
            "eval": "epoch+best_ckpt"},
    note="eval 도입. valid loss가 epoch1 0.251 최저 후 0.268/0.321로 상승 -> 과적합 확인, best=epoch1 자동 채택")
tracker.finish(cer_before, cer_after2, refs_a2, preds_b, preds_a2)
tracker.report()

[exp] exp2_eval_bestckpt 시작
      config: {"base": "seastar105/whisper-medium-komixv2", "lora_r": 32, "target": "q_proj,v_proj", "lr": 0.001, "epochs": 3, "train_n": 8000, "eval": "epoch+best_ckpt"}

  실험      : exp2_eval_bestckpt
  소요      : 0.0분
  CER       : 11.7%  ->  6.72%  (42.6% 개선)
  오류 분해 : 구두점 0.46%p / 숫자표기 -0.01%p / 띄어쓰기 0.33%p
              순수 인식 오류 5.94%
  샘플      : 개선 125건 / 퇴화 29건
리포트 생성 완료: ./experiments/REPORT.md


'./experiments/REPORT.md'

In [22]:
# =====================================================================
# [로그 정리] exp2가 두 번 실행돼 중복 기록됨 - 마지막 것만 남긴다
# =====================================================================
import json
from pathlib import Path

log = Path("./experiments/log.jsonl")
rows = [json.loads(l) for l in log.read_text(encoding="utf-8").strip().split("\n") if l]

# 같은 name이 여러 개면 마지막 것만 유지
seen, keep = set(), []
for r in reversed(rows):
    if r["name"] in seen:
        continue
    seen.add(r["name"])
    keep.append(r)
keep.reverse()

log.write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in keep) + "\n",
               encoding="utf-8")
print(f"{len(rows)}건 -> {len(keep)}건")
for r in keep:
    print(f"  {r['name']}: {r['cer_before']}% -> {r['cer_after']}%")

3건 -> 2건
  exp1_lora_r32_ep3: 11.7% -> 6.92%
  exp2_eval_bestckpt: 11.7% -> 6.72%


In [23]:
# =====================================================================
# [exp3] beam search 적용 - 학습 없이 추론 설정만으로 개선 시도
# ---------------------------------------------------------------------
# 배경: exp1~2에서 순수 인식오류가 5.97% -> 5.94%로 정체.
#       학습 파라미터 조정으로는 한계에 도달했다고 판단.
# 가설: greedy 디코딩 대신 beam search를 쓰면 학습 없이도 CER이 내려간다.
#       (여러 후보 경로를 탐색해 더 그럴듯한 문장을 선택)
# 비용: 추론 시간 증가. 준실시간 요구가 있는 서비스에는 트레이드오프 존재.
# =====================================================================
def measure_cer_beam(model, n=EVAL_SAMPLES, num_beams=5, tag=""):
    """beam search를 적용한 CER 측정 (exp1~2와 동일한 valid 샘플 사용)"""
    model.eval()
    preds, refs = [], []
    for i in range(n):
        ex = data["valid"][i]
        inputs = processor(
            ex["audio"]["array"], sampling_rate=16000, return_tensors="pt"
        ).input_features.to("cuda", torch.float16)
        with torch.no_grad():
            ids = model.generate(inputs, max_new_tokens=200, num_beams=num_beams)
        preds.append(processor.batch_decode(ids, skip_special_tokens=True)[0].strip())
        refs.append(ex["text"].strip())
        if (i + 1) % 50 == 0:
            print(f"  {tag} {i+1}/{n} 진행...")
    return jiwer.cer(refs, preds), preds, refs

import time
t0 = time.time()
cer_after3, preds_a3, refs_a3 = measure_cer_beam(model2, num_beams=5, tag="[exp3]")
elapsed = time.time() - t0

print(f"\nexp3 CER = {cer_after3*100:.2f}%")
print(f"추론 시간 = {elapsed/60:.1f}분 (200샘플, beam=5)")

[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  [exp3] 50/200 진행...


[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  [exp3] 100/200 진행...


[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  [exp3] 150/200 진행...


[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  [exp3] 200/200 진행...

exp3 CER = 6.30%
추론 시간 = 2.1분 (200샘플, beam=5)


In [24]:
tracker.start("exp3_beam5",
    config={"base": BASE_MODEL, "lora_r": 32, "target": "q_proj,v_proj",
            "lr": LEARNING_RATE, "epochs": NUM_EPOCHS, "train_n": 8000,
            "eval": "epoch+best_ckpt", "num_beams": 5},
    note="exp2 모델 그대로, 추론만 beam search(5). 발화당 0.6초로 준실시간 청크 방식에 적용 가능")
tracker.finish(cer_before, cer_after3, refs_a3, preds_b, preds_a3)
tracker.report()

[exp] exp3_beam5 시작
      config: {"base": "seastar105/whisper-medium-komixv2", "lora_r": 32, "target": "q_proj,v_proj", "lr": 0.001, "epochs": 3, "train_n": 8000, "eval": "epoch+best_ckpt", "num_beams": 5}

  실험      : exp3_beam5
  소요      : 0.0분
  CER       : 11.7%  ->  6.3%  (46.2% 개선)
  오류 분해 : 구두점 0.42%p / 숫자표기 -0.02%p / 띄어쓰기 0.41%p
              순수 인식 오류 5.48%
  샘플      : 개선 130건 / 퇴화 25건
리포트 생성 완료: ./experiments/REPORT.md


'./experiments/REPORT.md'

In [25]:
# =====================================================================
# [exp4] LoRA 용량 확장 - 정체 원인이 모델 용량인지 검증
# ---------------------------------------------------------------------
# 배경: exp1~3에서 순수 인식오류가 5.97 -> 5.94%로 정체.
#       표기 규칙은 이미 학습됐고 남은 건 실제 음향 인식 오류.
# 가설: 학습 용량(rank 32, q/v 2개 모듈)이 부족해 더 못 배우는 것이라면
#       rank와 대상 모듈을 늘렸을 때 순수 인식오류가 내려가야 한다.
# 판정: 안 내려가면 용량 문제가 아니라 데이터/도메인 한계(noise floor)로 결론.
# 안전장치: best checkpoint 유지 (용량이 커지면 과적합도 빨라지므로)
# =====================================================================
from transformers import WhisperForConditionalGeneration, Seq2SeqTrainer, Seq2SeqTrainingArguments
from peft import LoraConfig, get_peft_model
import torch

model4 = WhisperForConditionalGeneration.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16).to("cuda")
model4.generation_config.language = "korean"
model4.generation_config.task = "transcribe"

# 변경점: rank 32 -> 64, 대상 모듈 2개 -> 4개 (attention 전체)
model4 = get_peft_model(model4, LoraConfig(
    r=64, lora_alpha=128,
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj"],
    lora_dropout=0.05, bias="none",
))
model4.print_trainable_parameters()   # exp2의 943만개와 비교

args4 = Seq2SeqTrainingArguments(
    output_dir="./whisper-lora-exp4",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    fp16=True,
    logging_steps=25,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    report_to=[],
    remove_unused_columns=False,
    label_names=["labels"],
)

trainer4 = Seq2SeqTrainer(
    model=model4, args=args4,
    train_dataset=data_proc["train"],
    eval_dataset=data_proc["valid"],
    data_collator=collator,
)
trainer4.train()
model4.save_pretrained("./whisper-lora-exp4/final")
print("exp4 학습 완료")

Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

trainable params: 37,748,736 || all params: 801,606,656 || trainable%: 4.7091


Epoch,Training Loss,Validation Loss
1,0.350660,0.359060
2,0.252124,0.341926
3,0.109246,0.337981


exp4 학습 완료


In [26]:
cer_after4, preds_a4, refs_a4 = measure_cer_beam(model4, num_beams=5, tag="[exp4]")
print(f"exp4 CER = {cer_after4*100:.2f}%")

tracker.start("exp4_capacity_r64_qkvo",
    config={"base": BASE_MODEL, "lora_r": 64, "target": "q,k,v,out",
            "lr": LEARNING_RATE, "epochs": NUM_EPOCHS, "train_n": 8000,
            "eval": "epoch+best_ckpt", "num_beams": 5},
    note="용량 병목 검증. rank32->64, 모듈 2->4개. 순수 인식오류가 내려가면 용량 문제, 아니면 도메인 noise floor로 결론")
tracker.finish(cer_before, cer_after4, refs_a4, preds_b, preds_a4)
tracker.report()

[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  [exp4] 50/200 진행...


[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  [exp4] 100/200 진행...


[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  [exp4] 150/200 진행...


[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  [exp4] 200/200 진행...
exp4 CER = 8.09%
[exp] exp4_capacity_r64_qkvo 시작
      config: {"base": "seastar105/whisper-medium-komixv2", "lora_r": 64, "target": "q,k,v,out", "lr": 0.001, "epochs": 3, "train_n": 8000, "eval": "epoch+best_ckpt", "num_beams": 5}

  실험      : exp4_capacity_r64_qkvo
  소요      : 0.0분
  CER       : 11.7%  ->  8.09%  (30.8% 개선)
  오류 분해 : 구두점 0.43%p / 숫자표기 -0.06%p / 띄어쓰기 0.16%p
              순수 인식 오류 7.57%
  샘플      : 개선 109건 / 퇴화 45건
  ⚠ 퇴화 비중 높음 - 과적합 의심 (epoch 축소/eval 도입 검토)
리포트 생성 완료: ./experiments/REPORT.md


'./experiments/REPORT.md'

In [28]:
# =====================================================================
# [exp5] 학습률 스케줄 개선 - 정체가 noise floor인지 under-training인지 판별
# ---------------------------------------------------------------------
# 근거가 된 관측:
#   exp2(lr 1e-3): valid loss가 epoch1(0.251)에 이미 최저 -> 너무 빨리 외움
#   exp4(lr 1e-3, 용량2배): 0.338에서 수렴 실패 -> 용량 대비 LR 과다
#   => 두 실험 모두 "LR이 크다"를 가리킴. 아직 천천히 학습시켜본 적이 없음.
#
# 가설: LR을 낮추고 warmup+cosine으로 안정화하면 exp2의 valid loss 최저치
#      (0.251)보다 더 내려가고, 순수 인식오류도 5.9% 아래로 내려간다.
#
# 판정 기준: valid loss 최저치가 0.251보다 낮은가 (이게 핵심 지표)
# 구성은 검증된 exp2(r32, q/v)를 유지하고 학습 스케줄만 변경
# =====================================================================
from transformers import WhisperForConditionalGeneration, Seq2SeqTrainer, Seq2SeqTrainingArguments
from peft import LoraConfig, get_peft_model
import torch

model5 = WhisperForConditionalGeneration.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16).to("cuda")
model5.generation_config.language = "korean"
model5.generation_config.task = "transcribe"

# exp2와 동일 구성 (용량은 이미 exp4에서 병목이 아님을 확인)
model5 = get_peft_model(model5, LoraConfig(
    r=32, lora_alpha=64,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05, bias="none",
))
model5.print_trainable_parameters()

args5 = Seq2SeqTrainingArguments(
    output_dir="./whisper-lora-exp5",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    # --- 변경점: 학습 스케줄 ---
    learning_rate=2e-4,              # 1e-3 -> 2e-4 (5배 낮춤)
    num_train_epochs=6,              # 천천히 배우므로 더 길게
    warmup_ratio=0.1,                # 초반 급격한 변화 방지
    lr_scheduler_type="cosine",      # 후반 미세조정
    # -------------------------
    fp16=True,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    report_to=[],
    remove_unused_columns=False,
    label_names=["labels"],
)

trainer5 = Seq2SeqTrainer(
    model=model5, args=args5,
    train_dataset=data_proc["train"],
    eval_dataset=data_proc["valid"],
    data_collator=collator,
)
trainer5.train()
model5.save_pretrained("./whisper-lora-exp5/final")
print("exp5 학습 완료")

Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 9,437,184 || all params: 773,295,104 || trainable%: 1.2204


Epoch,Training Loss,Validation Loss
1,0.233189,0.233092
2,0.207688,0.232253
3,0.144335,0.245387
4,0.082693,0.291887
5,0.040178,0.327800
6,0.027834,0.339359


exp5 학습 완료


In [29]:
# =====================================================================
# [exp5 평가] exp3과 동일 조건(beam=5)으로 측정 - 공정 비교를 위해
# =====================================================================
cer_after5, preds_a5, refs_a5 = measure_cer_beam(model5, num_beams=5, tag="[exp5]")
print(f"exp5 CER = {cer_after5*100:.2f}%   (현재 최고: exp3 6.30%)")

[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  [exp5] 50/200 진행...


[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  [exp5] 100/200 진행...


[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  [exp5] 150/200 진행...


[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  [exp5] 200/200 진행...
exp5 CER = 6.10%   (현재 최고: exp3 6.30%)


In [30]:
tracker.start("exp5_lr2e-4_cosine_ep6",
    config={"base": BASE_MODEL, "lora_r": 32, "target": "q_proj,v_proj",
            "lr": 2e-4, "epochs": 6, "train_n": 8000,
            "eval": "epoch+best_ckpt+warmup+cosine", "num_beams": 5},
    note="exp2/exp4 valid loss 곡선이 공통으로 LR 과다를 시사 -> LR 5배 인하. "
         "valid loss 최저 0.2323(exp2 0.251 대비 개선), CER 6.10%로 최고 기록. "
         "under-training 가설 확정. 단 epoch2가 최적이라 6epoch은 과다, 3epoch이면 충분")
tracker.finish(cer_before, cer_after5, refs_a5, preds_b, preds_a5)
tracker.report()

[exp] exp5_lr2e-4_cosine_ep6 시작
      config: {"base": "seastar105/whisper-medium-komixv2", "lora_r": 32, "target": "q_proj,v_proj", "lr": 0.0002, "epochs": 6, "train_n": 8000, "eval": "epoch+best_ckpt+warmup+cosine", "num_beams": 5}

  실험      : exp5_lr2e-4_cosine_ep6
  소요      : 0.0분
  CER       : 11.7%  ->  6.1%  (47.8% 개선)
  오류 분해 : 구두점 0.48%p / 숫자표기 -0.03%p / 띄어쓰기 0.46%p
              순수 인식 오류 5.19%
  샘플      : 개선 127건 / 퇴화 20건
리포트 생성 완료: ./experiments/REPORT.md


'./experiments/REPORT.md'

In [31]:
import shutil, os
shutil.make_archive("./whisper_lora_exp5_adapter", "zip", "./whisper-lora-exp5/final")
print(os.path.getsize("./whisper_lora_exp5_adapter.zip")/1024**2, "MB")

33.37820339202881 MB


In [32]:
# =====================================================================
# [exp6] LR 최적점 탐색 - 하강 추세가 계속되는지 확인
# ---------------------------------------------------------------------
# 관측된 추세:
#   lr 1e-3 -> valid 최저 0.251  (epoch 1)
#   lr 2e-4 -> valid 최저 0.2323 (epoch 2)   낮출수록 개선
# 가설: 1e-4로 더 낮추면 valid 최저가 0.2323보다 내려간다.
#      내려가지 않으면 2e-4 부근이 최적점이며 LR 축은 소진된 것.
# epoch: LR을 절반으로 낮췄으므로 수렴에 더 필요 -> 4로 설정
#        (exp5에서 6 epoch은 과다했음을 확인, epoch2가 최적이었음)
# =====================================================================
from transformers import WhisperForConditionalGeneration, Seq2SeqTrainer, Seq2SeqTrainingArguments
from peft import LoraConfig, get_peft_model
import torch

model6 = WhisperForConditionalGeneration.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16).to("cuda")
model6.generation_config.language = "korean"
model6.generation_config.task = "transcribe"

model6 = get_peft_model(model6, LoraConfig(
    r=32, lora_alpha=64,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05, bias="none",
))
model6.print_trainable_parameters()

args6 = Seq2SeqTrainingArguments(
    output_dir="./whisper-lora-exp6",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=1e-4,              # 변경점: 2e-4 -> 1e-4
    num_train_epochs=4,              # 6 -> 4 (exp5에서 뒷 epoch 낭비 확인)
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    fp16=True,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    report_to=[],
    remove_unused_columns=False,
    label_names=["labels"],
)

trainer6 = Seq2SeqTrainer(
    model=model6, args=args6,
    train_dataset=data_proc["train"],
    eval_dataset=data_proc["valid"],
    data_collator=collator,
)
trainer6.train()
model6.save_pretrained("./whisper-lora-exp6/final")
print("exp6 학습 완료")

Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trainable params: 9,437,184 || all params: 773,295,104 || trainable%: 1.2204


Epoch,Training Loss,Validation Loss
1,0.237488,0.235463
2,0.222092,0.232540
3,0.184437,0.238162
4,0.156335,0.246393


exp6 학습 완료


In [33]:
# =====================================================================
# [exp6 평가] exp3·exp5와 동일 조건(beam=5)으로 측정
# =====================================================================
cer_after6, preds_a6, refs_a6 = measure_cer_beam(model6, num_beams=5, tag="[exp6]")
print(f"exp6 CER = {cer_after6*100:.2f}%   (현재 최고: exp5 6.10%)")

[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  [exp6] 50/200 진행...


[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  [exp6] 100/200 진행...


[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  [exp6] 150/200 진행...


[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

  [exp6] 200/200 진행...
exp6 CER = 6.47%   (현재 최고: exp5 6.10%)


In [34]:
# =====================================================================
# [exp9] 디코딩 파라미터 탐색 - 학습 없이 추론 설정만 바꿔 비교
# ---------------------------------------------------------------------
# beam 수만 바꿔봤을 뿐 다른 생성 파라미터는 건드리지 않았다.
# length_penalty: 긴 문장 선호도. Whisper가 문장을 짧게 끊는 경향을 보정
# repetition_penalty: 같은 표현 반복 억제
# 모델은 현재 최고인 exp5(model5)를 고정하고 디코딩만 바꾼다.
# =====================================================================
def measure_cer_cfg(model, n=EVAL_SAMPLES, **gen_kwargs):
    model.eval()
    preds, refs = [], []
    for i in range(n):
        ex = data["valid"][i]
        inputs = processor(ex["audio"]["array"], sampling_rate=16000,
                           return_tensors="pt").input_features.to("cuda", torch.float16)
        with torch.no_grad():
            ids = model.generate(inputs, max_new_tokens=200, **gen_kwargs)
        preds.append(processor.batch_decode(ids, skip_special_tokens=True)[0].strip())
        refs.append(ex["text"].strip())
    return jiwer.cer(refs, preds)

configs = [
    {"num_beams": 5},                                  # exp5 기준 = 6.10%
    {"num_beams": 10},
    {"num_beams": 5, "length_penalty": 1.2},
    {"num_beams": 5, "length_penalty": 0.8},
    {"num_beams": 5, "repetition_penalty": 1.1},
]

import time
results = []
for cfg in configs:
    t0 = time.time()
    c = measure_cer_cfg(model5, **cfg)
    dt = (time.time() - t0) / 60
    results.append((cfg, c, dt))
    print(f"{str(cfg):52s} CER {c*100:5.2f}%  ({dt:.1f}분)")

best = min(results, key=lambda x: x[1])
print(f"\n최적: {best[0]}  CER {best[1]*100:.2f}%")

[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

{'num_beams': 5}                                     CER  6.10%  (2.1분)


[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

{'num_beams': 10}                                    CER  6.20%  (2.4분)


[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

{'num_beams': 5, 'length_penalty': 1.2}              CER  6.08%  (2.1분)


[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

{'num_beams': 5, 'length_penalty': 0.8}              CER  6.12%  (2.1분)


[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

{'num_beams': 5, 'repetition_penalty': 1.1}          CER  6.08%  (2.1분)

최적: {'num_beams': 5, 'length_penalty': 1.2}  CER 6.08%


In [35]:
import transformers, warnings
transformers.logging.set_verbosity_error()
warnings.filterwarnings("ignore")
print("경고 억제 완료")

경고 억제 완료


In [36]:
tracker.start("exp9_decoding_params",
    config={"base_model": "exp5", "num_beams": "5/10",
            "length_penalty": "0.8/1.2", "repetition_penalty": 1.1},
    note="디코딩 파라미터 탐색. 최적 6.08%(length_penalty 1.2)로 기준 beam5 6.10% 대비 0.02%p. "
         "200샘플 기준 측정 노이즈 수준이라 유의한 개선 아님 -> 디코딩 축 소진. "
         "동시에 200샘플 평가의 해상도 한계 확인 -> 최종 순위는 500샘플 재측정 필요")
tracker.finish(cer_before, 0.0608, refs_a5, preds_b, preds_a5)
tracker.report()

[exp] exp9_decoding_params 시작
      config: {"base_model": "exp5", "num_beams": "5/10", "length_penalty": "0.8/1.2", "repetition_penalty": 1.1}

  실험      : exp9_decoding_params
  소요      : 0.0분
  CER       : 11.7%  ->  6.08%  (48.0% 개선)
  오류 분해 : 구두점 0.48%p / 숫자표기 -0.03%p / 띄어쓰기 0.46%p
              순수 인식 오류 5.19%
  샘플      : 개선 127건 / 퇴화 20건
리포트 생성 완료: ./experiments/REPORT.md


'./experiments/REPORT.md'

In [37]:
# =====================================================================
# [exp7 준비] 16000건 데이터셋 로드 및 전처리
# valid는 기존과 동일한 500건 유지 -> exp1~9와 직접 비교 가능
# =====================================================================
from datasets import load_dataset, Audio

data16 = load_dataset("json", data_files={
    "train": f"{DATA_ROOT}/train_16k.jsonl",
    "valid": f"{DATA_ROOT}/valid.jsonl",
})
data16 = data16.map(to_abs)
data16 = data16.cast_column("audio", Audio(sampling_rate=16000))
data16_proc = data16.map(prepare, remove_columns=data16["train"].column_names)
print(data16_proc)

Generating train split: 0 examples [00:00, ? examples/s]

Generating valid split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_features', 'labels'],
        num_rows: 16000
    })
    valid: Dataset({
        features: ['input_features', 'labels'],
        num_rows: 500
    })
})


In [38]:
# =====================================================================
# [exp7] 데이터 2배 (8000 -> 16000) - 변수는 '데이터 양' 하나만
# ---------------------------------------------------------------------
# 설정은 exp5(lr 2e-4, warmup 0.1, cosine)를 그대로 유지.
# epoch 3: 데이터가 2배라 같은 epoch도 스텝 수가 2배이며,
#          exp5에서 epoch2 부근이 최적이었으므로 3이면 충분.
# 가설: 데이터 양이 병목이었다면 valid 최저가 exp5(0.2323)보다 내려간다.
# =====================================================================
from transformers import WhisperForConditionalGeneration, Seq2SeqTrainer, Seq2SeqTrainingArguments
from peft import LoraConfig, get_peft_model
import torch

model7 = WhisperForConditionalGeneration.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16).to("cuda")
model7.generation_config.language = "korean"
model7.generation_config.task = "transcribe"

model7 = get_peft_model(model7, LoraConfig(
    r=32, lora_alpha=64,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05, bias="none",
))
model7.print_trainable_parameters()

args7 = Seq2SeqTrainingArguments(
    output_dir="./whisper-lora-exp7",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=2e-4,
    num_train_epochs=3,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    fp16=True, logging_steps=50,
    eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss", greater_is_better=False,
    save_total_limit=2, report_to=[],
    remove_unused_columns=False, label_names=["labels"],
)

trainer7 = Seq2SeqTrainer(
    model=model7, args=args7,
    train_dataset=data16_proc["train"],
    eval_dataset=data16_proc["valid"],
    data_collator=collator,
)
trainer7.train()
model7.save_pretrained("./whisper-lora-exp7/final")
print("exp7 학습 완료")

Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

trainable params: 9,437,184 || all params: 773,295,104 || trainable%: 1.2204
{'loss': '2.726', 'grad_norm': '1.048', 'learning_rate': '3.267e-05', 'epoch': '0.05'}
{'loss': '1.56', 'grad_norm': '0.4535', 'learning_rate': '6.6e-05', 'epoch': '0.1'}
{'loss': '0.9279', 'grad_norm': '1.009', 'learning_rate': '9.933e-05', 'epoch': '0.15'}
{'loss': '0.2954', 'grad_norm': '0.5194', 'learning_rate': '0.0001327', 'epoch': '0.2'}
{'loss': '0.2444', 'grad_norm': '0.3562', 'learning_rate': '0.000166', 'epoch': '0.25'}
{'loss': '0.2608', 'grad_norm': '0.3309', 'learning_rate': '0.0001993', 'epoch': '0.3'}
{'loss': '0.2555', 'grad_norm': '0.3661', 'learning_rate': '0.0001998', 'epoch': '0.35'}
{'loss': '0.2526', 'grad_norm': '0.3012', 'learning_rate': '0.0001993', 'epoch': '0.4'}
{'loss': '0.2355', 'grad_norm': '0.3673', 'learning_rate': '0.0001985', 'epoch': '0.45'}
{'loss': '0.2353', 'grad_norm': '0.4224', 'learning_rate': '0.0001973', 'epoch': '0.5'}
{'loss': '0.2252', 'grad_norm': '0.4698', 'lea

In [39]:
cer_after7, preds_a7, refs_a7 = measure_cer_beam(model7, num_beams=5, tag="[exp7]")
print(f"exp7 CER = {cer_after7*100:.2f}%   (현재 최고: exp5 6.10%)")

  [exp7] 50/200 진행...
  [exp7] 100/200 진행...
  [exp7] 150/200 진행...
  [exp7] 200/200 진행...
exp7 CER = 6.02%   (현재 최고: exp5 6.10%)


In [40]:
tracker.start("exp7_data16k",
    config={"base": BASE_MODEL, "lora_r": 32, "target": "q_proj,v_proj",
            "lr": 2e-4, "epochs": 3, "train_n": 16000,
            "eval": "epoch+best_ckpt+warmup+cosine", "num_beams": 5},
    note="데이터 2배(8000->16000). valid 최저 0.2323->0.2263, CER 6.10->6.02%. "
         "데이터 축은 아직 소진되지 않음. 단 epoch1에서 최저 도달 후 바로 과적합 "
         "-> 데이터를 늘려도 epoch은 짧게 가야 함. 개선폭 0.08%p는 200샘플 기준 "
         "노이즈 가능성 있어 500샘플 재측정 필요")
tracker.finish(cer_before, cer_after7, refs_a7, preds_b, preds_a7)
tracker.report()

[exp] exp7_data16k 시작
      config: {"base": "seastar105/whisper-medium-komixv2", "lora_r": 32, "target": "q_proj,v_proj", "lr": 0.0002, "epochs": 3, "train_n": 16000, "eval": "epoch+best_ckpt+warmup+cosine", "num_beams": 5}

  실험      : exp7_data16k
  소요      : 0.0분
  CER       : 11.7%  ->  6.02%  (48.6% 개선)
  오류 분해 : 구두점 0.42%p / 숫자표기 -0.12%p / 띄어쓰기 0.41%p
              순수 인식 오류 5.3%
  샘플      : 개선 125건 / 퇴화 20건
리포트 생성 완료: ./experiments/REPORT.md


'./experiments/REPORT.md'

In [41]:
# =====================================================================
# [exp8] 용량 재검증 @ 최적 LR
# ---------------------------------------------------------------------
# 변경점: LoRA rank 32 -> 64, 대상 모듈 q·v -> q·k·v·out
# 고정: 데이터 16000, lr 2e-4, warmup 0.1, cosine (exp7과 동일)
# 안전장치: load_best_model_at_end -> 용량이 커지면 과적합도 빨라지므로
#           최적 시점을 자동 채택한다
# =====================================================================
from transformers import (WhisperForConditionalGeneration,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments)
from peft import LoraConfig, get_peft_model
import torch

model8 = WhisperForConditionalGeneration.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16).to("cuda")
model8.generation_config.language = "korean"
model8.generation_config.task = "transcribe"

model8 = get_peft_model(model8, LoraConfig(
    r=64, lora_alpha=128,                                      # 용량 2배
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj"], # 어텐션 전체
    lora_dropout=0.05, bias="none",
))
model8.print_trainable_parameters()   # exp7의 943만개(1.22%)와 비교

args8 = Seq2SeqTrainingArguments(
    output_dir="./whisper-lora-exp8",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=2e-4,              # exp5에서 확정된 최적값
    num_train_epochs=3,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    fp16=True,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    report_to=[],
    remove_unused_columns=False,
    label_names=["labels"],
)

trainer8 = Seq2SeqTrainer(
    model=model8,
    args=args8,
    train_dataset=data16_proc["train"],   # 16000건 (exp7과 동일)
    eval_dataset=data16_proc["valid"],    # 기존 500건
    data_collator=collator,
)

trainer8.train()
model8.save_pretrained("./whisper-lora-exp8/final")
print("exp8 학습 완료")

Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

trainable params: 37,748,736 || all params: 801,606,656 || trainable%: 4.7091
{'loss': '2.304', 'grad_norm': '0.7359', 'learning_rate': '3.267e-05', 'epoch': '0.05'}
{'loss': '1.147', 'grad_norm': '0.8295', 'learning_rate': '6.6e-05', 'epoch': '0.1'}
{'loss': '0.2794', 'grad_norm': '0.7435', 'learning_rate': '9.933e-05', 'epoch': '0.15'}
{'loss': '0.2646', 'grad_norm': '0.5867', 'learning_rate': '0.0001327', 'epoch': '0.2'}
{'loss': '0.2293', 'grad_norm': '0.4233', 'learning_rate': '0.000166', 'epoch': '0.25'}
{'loss': '0.2464', 'grad_norm': '0.4438', 'learning_rate': '0.0001993', 'epoch': '0.3'}
{'loss': '0.2347', 'grad_norm': '0.4301', 'learning_rate': '0.0001998', 'epoch': '0.35'}
{'loss': '0.2404', 'grad_norm': '0.4345', 'learning_rate': '0.0001993', 'epoch': '0.4'}
{'loss': '0.2231', 'grad_norm': '0.5347', 'learning_rate': '0.0001985', 'epoch': '0.45'}
{'loss': '0.2249', 'grad_norm': '0.499', 'learning_rate': '0.0001973', 'epoch': '0.5'}
{'loss': '0.2136', 'grad_norm': '0.6644', '

In [42]:
cer_after8, preds_a8, refs_a8 = measure_cer_beam(model8, num_beams=5, tag="[exp8]")
print(f"exp8 CER = {cer_after8*100:.2f}%   (현재 최고: exp7 6.02%)")


  [exp8] 50/200 진행...
  [exp8] 100/200 진행...
  [exp8] 150/200 진행...
  [exp8] 200/200 진행...
exp8 CER = 5.43%   (현재 최고: exp7 6.02%)


In [43]:
tracker.start("exp8_capacity_r64_at_lr2e-4",
    config={"base": BASE_MODEL, "lora_r": 64, "target": "q,k,v,out",
            "lr": 2e-4, "epochs": 3, "train_n": 16000,
            "eval": "epoch+best_ckpt+warmup+cosine", "num_beams": 5},
    note="exp4 재검증 결과 초기 결론이 뒤집힘. 동일 용량(r64,q·k·v·o)인데 LR만 1e-3->2e-4로 "
         "바꾸자 valid 0.338->0.2232, CER 8.09%->5.43%. exp4의 실패 원인은 용량이 아니라 LR "
         "과다였음이 확정. 용량 축은 유효하며 아직 소진되지 않음. 단 파라미터 4.71%로 exp7의 "
         "4배라 epoch1에서 이미 최저 -> 학습은 짧게")
tracker.finish(cer_before, cer_after8, refs_a8, preds_b, preds_a8)
tracker.report()

[exp] exp8_capacity_r64_at_lr2e-4 시작
      config: {"base": "seastar105/whisper-medium-komixv2", "lora_r": 64, "target": "q,k,v,out", "lr": 0.0002, "epochs": 3, "train_n": 16000, "eval": "epoch+best_ckpt+warmup+cosine", "num_beams": 5}

  실험      : exp8_capacity_r64_at_lr2e-4
  소요      : 0.0분
  CER       : 11.7%  ->  5.43%  (53.6% 개선)
  오류 분해 : 구두점 0.33%p / 숫자표기 -0.16%p / 띄어쓰기 0.41%p
              순수 인식 오류 4.85%
  샘플      : 개선 133건 / 퇴화 15건
리포트 생성 완료: ./experiments/REPORT.md


'./experiments/REPORT.md'

In [44]:
# =====================================================================
# [exp11] 용량 축 한 단계 더 - r64에서 개선됐으니 r128은 어떤가
# ---------------------------------------------------------------------
# 근거: exp8에서 r32->r64로 CER 6.02->5.43% (지금까지 최대 단일 개선폭).
#       용량 축이 아직 살아있으므로 한 단계 더 확인한다.
# 변경점: rank 64 -> 128 만. LR·데이터·모듈은 exp8과 동일하게 고정.
#         (exp4의 교훈: 한 번에 두 변수를 바꾸면 원인을 알 수 없다)
# epoch 2: exp8이 파라미터 4.71%로 epoch1에 이미 최저를 찍었고,
#          r128은 그보다 2배라 더 빨리 과적합한다. 3 epoch은 낭비.
# 주의: 만약 r128이 나빠지면 exp4처럼 'LR 대비 용량 과다'일 수 있다.
#       그 경우 다음 실험은 r128 + 더 낮은 LR이 된다.
# =====================================================================
from transformers import (WhisperForConditionalGeneration,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments)
from peft import LoraConfig, get_peft_model
import torch

model11 = WhisperForConditionalGeneration.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16).to("cuda")
model11.generation_config.language = "korean"
model11.generation_config.task = "transcribe"

model11 = get_peft_model(model11, LoraConfig(
    r=128, lora_alpha=256,                                     # 변경점
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj"], # exp8과 동일
    lora_dropout=0.05, bias="none",
))
model11.print_trainable_parameters()   # exp8의 3,775만(4.71%)과 비교

args11 = Seq2SeqTrainingArguments(
    output_dir="./whisper-lora-exp11",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=2e-4,
    num_train_epochs=2,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    fp16=True, logging_steps=50,
    eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss", greater_is_better=False,
    save_total_limit=2, report_to=[],
    remove_unused_columns=False, label_names=["labels"],
)

trainer11 = Seq2SeqTrainer(
    model=model11, args=args11,
    train_dataset=data16_proc["train"],
    eval_dataset=data16_proc["valid"],
    data_collator=collator,
)
trainer11.train()
model11.save_pretrained("./whisper-lora-exp11/final")
print("exp11 학습 완료")

Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]

trainable params: 75,497,472 || all params: 839,355,392 || trainable%: 8.9947
{'loss': '1.961', 'grad_norm': '0.6969', 'learning_rate': '4.9e-05', 'epoch': '0.05'}
{'loss': '0.6166', 'grad_norm': '0.5983', 'learning_rate': '9.9e-05', 'epoch': '0.1'}
{'loss': '0.2413', 'grad_norm': '0.8336', 'learning_rate': '0.000149', 'epoch': '0.15'}
{'loss': '0.257', 'grad_norm': '0.6191', 'learning_rate': '0.000199', 'epoch': '0.2'}
{'loss': '0.2278', 'grad_norm': '0.5263', 'learning_rate': '0.0001996', 'epoch': '0.25'}
{'loss': '0.2412', 'grad_norm': '0.5349', 'learning_rate': '0.0001985', 'epoch': '0.3'}
{'loss': '0.2388', 'grad_norm': '0.5242', 'learning_rate': '0.0001966', 'epoch': '0.35'}
{'loss': '0.2432', 'grad_norm': '0.62', 'learning_rate': '0.000194', 'epoch': '0.4'}
{'loss': '0.2238', 'grad_norm': '0.6752', 'learning_rate': '0.0001907', 'epoch': '0.45'}
{'loss': '0.2255', 'grad_norm': '0.68', 'learning_rate': '0.0001867', 'epoch': '0.5'}
{'loss': '0.2128', 'grad_norm': '0.8712', 'learnin

In [47]:
# =====================================================================
# [exp10] 상위 후보 500샘플 재측정 - 최종 순위 확정
# ---------------------------------------------------------------------
# 지금까지 모든 CER은 valid 200샘플 기준이다. exp9에서 0.02%p 차이가
# 노이즈임을 확인했듯, 200샘플은 미세한 차이를 구분할 해상도가 없다.
# 최종 후보만 valid 전체(500)로 다시 재서 순위를 확정한다.
# =====================================================================
FINALISTS = {
    "exp5  (r32, 8k)":   model5,
    "exp7  (r32, 16k)":  model7,
    "exp8  (r64, 16k)":  model8,
    "exp11 (r128, 16k)": model11,
}

print(f"{'모델':22s} {'200샘플':>10s} {'500샘플':>10s}")
final = {}
for name, m in FINALISTS.items():
    c500, _, _ = measure_cer_beam(m, n=500, num_beams=5, tag=f"[{name}]")
    final[name] = c500
    print(f"{name:22s} {'-':>10s} {c500*100:9.2f}%")

best = min(final, key=final.get)
print(f"\n최종 채택: {best}  CER {final[best]*100:.2f}%")
print(f"baseline 11.70% 대비 {(0.117-final[best])/0.117*100:.1f}% 개선")

모델                          200샘플      500샘플
  [exp5  (r32, 8k)] 50/500 진행...
  [exp5  (r32, 8k)] 100/500 진행...
  [exp5  (r32, 8k)] 150/500 진행...
  [exp5  (r32, 8k)] 200/500 진행...
  [exp5  (r32, 8k)] 250/500 진행...
  [exp5  (r32, 8k)] 300/500 진행...
  [exp5  (r32, 8k)] 350/500 진행...
  [exp5  (r32, 8k)] 400/500 진행...
  [exp5  (r32, 8k)] 450/500 진행...
  [exp5  (r32, 8k)] 500/500 진행...
exp5  (r32, 8k)                 -      6.05%
  [exp7  (r32, 16k)] 50/500 진행...
  [exp7  (r32, 16k)] 100/500 진행...
  [exp7  (r32, 16k)] 150/500 진행...
  [exp7  (r32, 16k)] 200/500 진행...
  [exp7  (r32, 16k)] 250/500 진행...
  [exp7  (r32, 16k)] 300/500 진행...
  [exp7  (r32, 16k)] 350/500 진행...
  [exp7  (r32, 16k)] 400/500 진행...
  [exp7  (r32, 16k)] 450/500 진행...
  [exp7  (r32, 16k)] 500/500 진행...
exp7  (r32, 16k)                -      5.86%
  [exp8  (r64, 16k)] 50/500 진행...
  [exp8  (r64, 16k)] 100/500 진행...
  [exp8  (r64, 16k)] 150/500 진행...
  [exp8  (r64, 16k)] 200/500 진행...
  [exp8  (r64, 16k)] 250/500 진행...
  [

In [46]:
cer_after11, preds_a11, refs_a11 = measure_cer_beam(model11, num_beams=5, tag="[exp11]")
print(f"exp11 CER = {cer_after11*100:.2f}%   (현재 최고: exp8 5.43%)")

  [exp11] 50/200 진행...
  [exp11] 100/200 진행...
  [exp11] 150/200 진행...
  [exp11] 200/200 진행...
exp11 CER = 5.38%   (현재 최고: exp8 5.43%)


In [3]:
import sys, os
from pathlib import Path

PROJ = str(Path.home() / "stt-finetune")
if PROJ not in sys.path:
    sys.path.insert(0, PROJ)

from exp_tracker import ExpTracker
tracker = ExpTracker()   # ./experiments/ 에 있는 기존 log.jsonl에 이어서 기록됨

In [4]:
print(len(tracker.history()), "건 기록됨")

8 건 기록됨


In [6]:
# =====================================================================
# [exp11 기록 - 수동] 커널 재시작으로 변수가 소실되어 수치만 직접 기록
# ---------------------------------------------------------------------
import json, datetime
from pathlib import Path

rec = {
    "time": datetime.datetime.now().strftime("%Y-%m-%d %H:%M"),
    "name": "exp11_capacity_r128",
    "config": {"lora_r": 128, "target": "q,k,v,out", "lr": 2e-4,
               "epochs": 2, "train_n": 16000, "num_beams": 5},
    "started": "2026-07-24",
    "minutes": 49,
    "cer_before": 11.70,
    "cer_after": 5.74,          # 500샘플 최종 수치
    "improve_pct": round((11.70-5.74)/11.70*100, 1),
    "note": ("용량 축 한 단계 더. 200샘플 5.38%로 exp8(5.43%)보다 나아 보였으나 "
             "500샘플 재측정 시 5.74% vs 5.56%로 역전. 용량은 r64에서 포화하며 "
             "r128은 파라미터 2배를 쓰고 오히려 과적합. 표본 부족으로 인한 순위 오판 사례."),
}

log = Path("./experiments/log.jsonl")
with open(log, "a", encoding="utf-8") as f:
    f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print("exp11 기록 완료")
tracker.report()

exp11 기록 완료
리포트 생성 완료: ./experiments/REPORT.md


'./experiments/REPORT.md'

In [7]:
for line in log.read_text(encoding="utf-8").strip().split("\n"):
    r = json.loads(line)
    print(f"{r['name']:28s} {r['cer_before']}% -> {r['cer_after']}%")

exp1_lora_r32_ep3            11.7% -> 6.92%
exp2_eval_bestckpt           11.7% -> 6.72%
exp3_beam5                   11.7% -> 6.3%
exp4_capacity_r64_qkvo       11.7% -> 8.09%
exp5_lr2e-4_cosine_ep6       11.7% -> 6.1%
exp9_decoding_params         11.7% -> 6.08%
exp7_data16k                 11.7% -> 6.02%
exp8_capacity_r64_at_lr2e-4  11.7% -> 5.43%
exp11_capacity_r128          11.7% -> 5.74%


In [8]:
import json, datetime
from pathlib import Path

rec = {
    "time": datetime.datetime.now().strftime("%Y-%m-%d %H:%M"),
    "name": "exp6_lr1e-4_cosine_ep4",
    "config": {"lora_r": 32, "target": "q_proj,v_proj", "lr": 1e-4,
               "epochs": 4, "train_n": 8000, "eval": "epoch+best_ckpt+warmup+cosine", "num_beams": 5},
    "started": "2026-07-24",
    "minutes": 45,
    "cer_before": 11.70,
    "cer_after": 6.47,
    "improve_pct": round((11.70-6.47)/11.70*100, 1),
    "note": ("LR 최적점 탐색. 2e-4 -> 1e-4로 인하했으나 CER 6.47%로 악화(exp5 6.10%). "
             "valid 최저는 0.2325로 exp5(0.2323)와 거의 동일하나 train loss가 0.156에서 "
             "멈춰 학습 부족 상태. 2e-4가 최적점이며 LR 축 소진 확인."),
}

log = Path("./experiments/log.jsonl")
with open(log, "a", encoding="utf-8") as f:
    f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print("exp6 기록 완료")
tracker.report()

exp6 기록 완료
리포트 생성 완료: ./experiments/REPORT.md


'./experiments/REPORT.md'

In [9]:
rec = {
    "time": datetime.datetime.now().strftime("%Y-%m-%d %H:%M"),
    "name": "exp10_final_500sample_reeval",
    "config": {"candidates": "exp5,exp7,exp8,exp11", "n_samples": 500, "num_beams": 5},
    "started": "2026-07-24",
    "minutes": 20,
    "cer_before": 11.70,
    "cer_after": 5.56,
    "improve_pct": round((11.70-5.56)/11.70*100, 1),
    "note": ("최종 후보 4개(exp5/7/8/11)를 valid 전체 500샘플로 재측정. "
             "결과: 6.05% / 5.86% / 5.56% / 5.74%. 200샘플 순위(exp11<exp8)가 "
             "500샘플에서 역전(exp8<exp11) -> exp8 최종 채택. "
             "표본 부족이 순위를 오판시킬 수 있음을 확인한 검증 단계."),
}
with open(log, "a", encoding="utf-8") as f:
    f.write(json.dumps(rec, ensure_ascii=False) + "\n")
print("exp10 기록 완료")
tracker.report()

exp10 기록 완료
리포트 생성 완료: ./experiments/REPORT.md


'./experiments/REPORT.md'

In [10]:
for line in log.read_text(encoding="utf-8").strip().split("\n"):
    r = json.loads(line)
    print(f"{r['name']:32s} {r['cer_before']}% -> {r['cer_after']}%")

exp1_lora_r32_ep3                11.7% -> 6.92%
exp2_eval_bestckpt               11.7% -> 6.72%
exp3_beam5                       11.7% -> 6.3%
exp4_capacity_r64_qkvo           11.7% -> 8.09%
exp5_lr2e-4_cosine_ep6           11.7% -> 6.1%
exp9_decoding_params             11.7% -> 6.08%
exp7_data16k                     11.7% -> 6.02%
exp8_capacity_r64_at_lr2e-4      11.7% -> 5.43%
exp11_capacity_r128              11.7% -> 5.74%
exp6_lr1e-4_cosine_ep4           11.7% -> 6.47%
exp10_final_500sample_reeval     11.7% -> 5.56%


In [11]:
import shutil, os

shutil.make_archive("./experiments_backup", "zip", "./experiments")
print(f"experiments_backup.zip: {os.path.getsize('./experiments_backup.zip')/1024**2:.1f} MB")

shutil.make_archive("./whisper_lora_exp8_adapter", "zip", "./whisper-lora-exp8/final")
print(f"whisper_lora_exp8_adapter.zip: {os.path.getsize('./whisper_lora_exp8_adapter.zip')/1024**2:.1f} MB")

print("\n파일탐색기에서 두 zip 모두 우클릭 → Download")

experiments_backup.zip: 0.0 MB
whisper_lora_exp8_adapter.zip: 133.3 MB

파일탐색기에서 두 zip 모두 우클릭 → Download
